# <font color="#418FDE" size="6.5" uppercase>**B: Object Detection Experiments**</font>
----

> Last update: 20240827

By the end of this lecture, you will be able to:

* Develop fine-tuned object detection models such as Faster RCNN & YOLO.


## **1. Experiment with Pascal Visual Object Classes (VOC) Dataset & Faster RCNN**

### **1.1. Pascal VOC Dataset**

> The Pascal Visual Object Classes (VOC)  dataset is part of a series of challenges & datasets organized by the Pascal Network of Excellence, which was a European Union-funded research initiative focused on pattern analysis, statistical modeling, & computational learning. The VOC challenge is one of the most well-known benchmarks in computer vision, particularly for object detection & segmentation tasks.

> The Pascal VOC challenges, particularly those held from 2005 to 2012, have influenced object detection, image classification, & segmentation research. The datasets provided by these challenges are still widely used today for training & evaluating computer vision models.  

> The Pascal VOC dataset comes in different versions, primarily the 2007 & 2012 editions, each with a different number of images:
- **VOC 2007**:
  - It contains approximately **9,963** images.
  - These images are split into training, validation, & test sets:
    - Training & validation combined: 5,011 images
    - Test set: 4,952 images
- **VOC 2012**:
  - It contains approximately **11,540** images.
  - These images are split into training, validation, & test sets:
    - Training & validation combined: 5,717 images
    - Test set: 5,823 images

> These images are annotated with bounding boxes & class labels for 20 object categories, making Pascal VOC a popular dataset for training & evaluating object detection models. Let's download & observe samples of this dataset for the object detection tasks.

In [ ]:
#@title Import Necessary Libraries
'''
Runtime: CPU

torch:                              https://pytorch.org/docs/stable/index.html
torchvision:                        https://pytorch.org/vision/stable/index.html
torchvision.datasets.VOCDetection:  https://pytorch.org/vision/stable/datasets.html#torchvision.datasets.VOCDetection
torch.utils.data.DataLoader:        https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader
matplotlib.pyplot:                  https://matplotlib.org/stable/api/pyplot_api.html
numpy:                              https://numpy.org/doc/stable/
PIL.Image:                          https://pillow.readthedocs.io/en/stable/reference/Image.html
PIL.ImageDraw:                      https://pillow.readthedocs.io/en/stable/reference/ImageDraw.html
'''

# Importing the PyTorch library, which is widely used for deep learning tasks
import torch

# Importing the torchvision library, which provides popular datasets,
# model architectures, & image transformations for computer vision
import torchvision

# Importing the transforms module from torchvision, which provides common image
# transformations that can be applied to images
from torchvision import transforms

# Importing the VOCDetection dataset class from torchvision.datasets. This is a
# dataset class for the Pascal VOC dataset, which is often used for object
# detection tasks
from torchvision.datasets import VOCDetection

# Importing the DataLoader class from torch.utils.data. DataLoader is used to
# load data in batches, shuffle the data, & handle multi-threaded data loading,
# which is crucial for efficient training
from torch.utils.data import DataLoader

# Importing matplotlib's pyplot module for creating plots & visualizing data
import matplotlib.pyplot as plt

# Importing numpy, a powerful library for numerical computations, often used in
# conjunction with PyTorch for handling arrays & matrices
import numpy as np

# Importing specific modules from the Python Imaging Library (PIL), which
# provides extensive file format support, an efficient internal
# representation, & powerful image processing capabilities
from PIL import Image, ImageDraw




In [ ]:
#@title Load the Data

# Define the transformation to convert images to tensor
# In PyTorch, images are typically represented as tensors.
# This transformation will convert the images loaded from the dataset into a PyTorch tensor format.
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts a PIL Image or numpy array to a tensor.
])

# Load the Pascal VOC dataset
voc_dataset = VOCDetection(
    root='./voc',  # Directory where the VOC data will be stored. This is where the dataset will be downloaded & saved.
    year='2007',  # Specifies the year of the VOC dataset to be used (either 2007 or 2012).
    image_set='train',  # Specifies the subset of the dataset to be used ('train', 'val', or 'trainval').
    download=True,  # If set to True, the dataset will be downloaded from the internet if it’s not already available in the root directory.
    transform=transform  # The transformation to apply to each image. Here, it converts the images to tensors.
)


In [ ]:
#@title Custom Collate Function & Data Loader

# Custom collate function to handle images of different sizes
# The default collate function in PyTorch assumes all images in a batch are the same size.
# This custom function will handle cases where images in a batch have different sizes.
def custom_collate_fn(batch):
    images = [item[0] for item in batch]  # Extract the images from the batch.
    annotations = [item[1] for item in batch]  # Extract the annotations (bounding boxes, labels) from the batch.
    return images, annotations  # Return the images & annotations as separate lists.

# Create a DataLoader with the custom collate function
# A `DataLoader` in PyTorch is a powerful utility that handles the loading,
# batching, & optional shuffling of datasets, making the data pipeline efficient
# & streamlined during training & evaluation. It allows you to load data in
# manageable batches, which is crucial for leveraging GPU acceleration, & can
# also handle complex tasks like shuffling data to improve model generalization
# & managing multiple workers for parallel data loading, all of which are
# essential for training machine learning models effectively.
data_loader = DataLoader(
    voc_dataset,  # The dataset to load (in this case, the Pascal VOC dataset).
    batch_size=4,  # Number of samples (images) per batch.
    shuffle=True,  # Whether to shuffle the dataset after each epoch (good for training to prevent the model from learning the order of data).
    collate_fn=custom_collate_fn  # Use the custom collate function to handle varying image sizes.
)

In [ ]:
#@title Data Visualization

# Function to visualize a sample from the dataset
# This function will take an image & its corresponding annotations & display the image with bounding boxes around detected objects.
def visualize_sample(img, anns):
    # Convert tensor to PIL image for visualization
    img = transforms.ToPILImage()(img)  # Convert the tensor back to a PIL image, which can be easily displayed.

    # Draw bounding boxes on the image
    draw = ImageDraw.Draw(img)  # Create an object that can draw on the image.
    for ann in anns['annotation']['object']:  # Loop over each object in the annotation.
        bbox = ann['bndbox']  # Extract the bounding box coordinates.
        x_min, y_min = int(bbox['xmin']), int(bbox['ymin'])  # Get the top-left corner of the bounding box.
        x_max, y_max = int(bbox['xmax']), int(bbox['ymax'])  # Get the bottom-right corner of the bounding box.
        draw.rectangle([x_min, y_min, x_max, y_max], outline='red', width=2)  # Draw the bounding box on the image.
        category_name = ann['name']  # Get the category (label) of the object.
        draw.text((x_min, y_min), category_name, fill='white')  # Draw the category name near the bounding box.

    plt.figure(figsize = [5,5])
    plt.imshow(np.asarray(img))  # Display the image with bounding boxes using matplotlib.
    plt.axis('off')  # Turn off the axis for a cleaner look.
    plt.show()  # Show the image in a new window.

# Iterate through the data loader & visualize the first batch
# The following loop will go through one batch of images & annotations, visualize each image with its annotations, & then stop.
for imgs, annotations in data_loader:
    for i in range(2):  # Loop over the first two images in the batch.
        visualize_sample(imgs[i], annotations[i])  # Visualize the image & its annotations.
    break  # Stop after visualizing the first batch for simplicity.

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()

### **1.2. Training Using Faster RCNN**


> Training the Pascal VOC dataset with Faster R-CNN involves leveraging one of the most powerful & widely used object detection models in deep learning. Faster R-CNN, which combines region proposal networks (RPN) with a Fast R-CNN detector, enables efficient & accurate detection of objects within images. The model is typically initialized with pre-trained weights, such as those from training on the COCO dataset, to leverage transfer learning. This approach helps the model start with a strong understanding of basic visual features, speeding up convergence & improving performance on the VOC dataset, which contains 20 object classes. During training, the model learns to predict bounding boxes & classify objects within these boxes, refining its predictions with each epoch as it minimizes the loss function, which typically combines classification & regression losses.

> The training process requires careful handling of hyperparameters like learning rate, batch size, & the number of epochs. These parameters significantly influence the model's performance & its ability to generalize to unseen data. Additionally, techniques like data augmentation & learning rate scheduling are often employed to improve robustness & avoid overfitting. The evaluation phase, following training, involves testing the model on a separate validation set to ensure it has learned to accurately detect & classify objects. The Faster R-CNN model's performance on VOC is typically measured using metrics such as mean Average Precision (mAP), which reflects the precision & recall of the model across different object categories. Overall, training Faster R-CNN on the Pascal VOC dataset provides a strong foundation for building & fine-tuning object detection systems in practical applications.

In [ ]:
#@title Import Necessary Libraries
'''
Runtime: GPU $$$

torch:                                    https://pytorch.org/docs/stable/index.html
torchvision:                              https://pytorch.org/vision/stable/index.html
torchvision.transforms:                   https://pytorch.org/vision/0.9/transforms.html
torchvision.ops.nms:                      https://pytorch.org/vision/main/generated/torchvision.ops.nms.html
torchvision.models.detection.FasterRCNN:  https://pytorch.org/vision/stable/models.html#torchvision.models.detection.FasterRCNN
torchvision.transforms.functional:        https://pytorch.org/vision/stable/transforms.html#functional-transforms
torchvision.datasets.VOCDetection:        https://pytorch.org/vision/stable/datasets.html#torchvision.datasets.VOCDetection
torch.utils.data.DataLoader:              https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader
numpy:                                    https://numpy.org/doc/stable/
matplotlib.pyplot:                        https://matplotlib.org/stable/api/pyplot_api.html
PIL.Image:                                https://pillow.readthedocs.io/en/stable/reference/Image.html
PIL.ImageDraw:                            https://pillow.readthedocs.io/en/stable/reference/ImageDraw.html
'''

# Importing the PyTorch library, which is widely used for deep learning tasks.
import torch

# Importing the torchvision library, which provides popular datasets,
# model architectures, & image transformations for computer vision tasks.
import torchvision

# Importing the transforms module from torchvision, which provides common image
# transformations that can be applied to images
from torchvision import transforms

# Import Non-Maximum Suppression (NMS) from torchvision
from torchvision.ops import nms

# Importing the Faster R-CNN model from torchvision's detection module.
# Faster R-CNN is a popular object detection model that predicts bounding boxes
# & labels for objects in images.
from torchvision.models.detection import FasterRCNN

# Importing the AnchorGenerator from torchvision's detection module.
# The AnchorGenerator is responsible for generating anchor boxes of different
# sizes & aspect ratios in the RPN (Region Proposal Network) of Faster R-CNN.
from torchvision.models.detection.rpn import AnchorGenerator

# Importing the functional API for image transformations from torchvision.transforms.
# The functional API provides a set of utilities for applying transformations
# (like resizing, cropping, etc.) directly to images.
from torchvision.transforms import functional as F

# Importing the VOCDetection dataset class from torchvision.datasets.
# VOCDetection is a dataset class for the Pascal VOC dataset, which is widely
# used for object detection tasks.
from torchvision.datasets import VOCDetection

# Importing the DataLoader class from torch.utils.data.
# DataLoader is used to load data in batches, shuffle the data, & handle
# multi-threaded data loading, which is crucial for efficient training & testing.
from torch.utils.data import DataLoader

# Importing the Dataset class from torch.utils.data.
# The Dataset class is a base class for all datasets in PyTorch. It allows you
# to create custom datasets by inheriting from this class.
from torch.utils.data import Dataset

# Importing the NumPy library, which is a powerful library for numerical computations in Python.
# NumPy is often used in conjunction with PyTorch for handling arrays & matrices.
import numpy as np

# Importing the pyplot module from matplotlib, which is a popular library for
# creating static, animated, & interactive visualizations in Python.
# Pyplot provides a MATLAB-like interface for making plots.
import matplotlib.pyplot as plt

# Importing specific modules from the Python Imaging Library (PIL), which provides extensive file format support,
# an efficient internal representation, & powerful image processing capabilities.
from PIL import Image, ImageDraw

# Import tqdm for progress bar
from tqdm import tqdm

# Set the device to GPU if available, otherwise CPU
# The torch.device object represents the device on which a tensor will be allocated (either CPU or GPU).
# 'cuda' refers to GPU, & 'cpu' refers to the CPU.
# If a GPU is available, the model & data will be moved to the GPU for faster
# computation. Otherwise, they will run on the CPU.
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')


In [ ]:
#@title Custom Dataset Class

# Define the class name to index mapping for Pascal VOC
VOC_CLASSES = {
    'aeroplane': 1, 'bicycle': 2, 'bird': 3, 'boat': 4,
    'bottle': 5, 'bus': 6, 'car': 7, 'cat': 8, 'chair': 9,
    'cow': 10, 'diningtable': 11, 'dog': 12, 'horse': 13,
    'motorbike': 14, 'person': 15, 'pottedplant': 16,
    'sheep': 17, 'sofa': 18, 'train': 19, 'tvmonitor': 20
}

# Define a custom dataset class named VOCDataset that inherits from torch.utils.data.Dataset.
# This class will handle the loading & preprocessing of the Pascal VOC dataset for object detection tasks.
class VOCDataset(Dataset):

    # The __init__ method is the constructor for the class. It initializes the dataset.
    def __init__(self, root, year, image_set, transforms=None):
        # Load the Pascal VOC dataset using the VOCDetection class from torchvision.
        # - root: The directory where the VOC data will be stored or is already stored.
        # - year: The year of the VOC dataset to use (e.g., '2007' or '2012').
        # - image_set: Specifies whether to load the 'train', 'val', or 'test' split of the dataset.
        # - download=True: Automatically downloads the dataset if it's not already available in the root directory.
        self.dataset = VOCDetection(root=root, year=year, image_set=image_set, download=True)

        # Store the optional image transformations (e.g., resizing, normalizing) to be applied to each image.
        self.transforms = transforms

    # The __getitem__ method retrieves a single sample (image & its corresponding target/label) from the dataset.
    # - idx: The index of the sample to retrieve.
    def __getitem__(self, idx):
        # Get the image & its corresponding annotations (bounding boxes, labels) from the VOC dataset.
        img = self.dataset[idx][0]  # Retrieve the image.
        target = self.dataset[idx][1]['annotation']  # Retrieve the annotations associated with the image.

        # Initialize empty lists to store bounding boxes & labels.
        boxes = []
        labels = []

        # Extract bounding boxes & labels from the annotation.
        for obj in target['object']:
            bbox = obj['bndbox']  # Get the bounding box coordinates from the annotation.
            # Convert the bounding box coordinates to integers & store them in the boxes list.
            boxes.append([int(bbox['xmin']), int(bbox['ymin']), int(bbox['xmax']), int(bbox['ymax'])])
            # Map the object class name to its corresponding integer label using the VOC_CLASSES dictionary.
            labels.append(VOC_CLASSES[obj['name']])

        # Convert the lists of bounding boxes & labels into PyTorch tensors.
        boxes = torch.as_tensor(boxes, dtype=torch.float32)  # Bounding boxes as a float tensor.
        labels = torch.as_tensor(labels, dtype=torch.int64)  # Labels as an integer tensor.

        # Create a tensor containing the index of the image (used as a unique identifier).
        image_id = torch.tensor([idx])

        # Calculate the area of each bounding box (used for evaluation & training).
        # - The area is calculated as (width * height) for each bounding box.
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])

        # Create a tensor to indicate whether an object is "crowded" (for now, we set all to 0, meaning not crowded).
        # - The "iscrowd" field is typically used in COCO-style datasets, where some objects might be annotated as being in a crowd.
        # - In Pascal VOC, this is generally set to 0.
        iscrowd = torch.zeros((len(labels),), dtype=torch.int64)

        # Create a dictionary to store the target information (bounding boxes, labels, etc.).
        target = {}
        target['boxes'] = boxes  # Add bounding boxes to the target.
        target['labels'] = labels  # Add labels to the target.
        target['image_id'] = image_id  # Add image ID to the target.
        target['area'] = area  # Add area of bounding boxes to the target.
        target['iscrowd'] = iscrowd  # Add iscrowd information to the target.

        # Apply the optional transformations to the image if they were provided.
        if self.transforms:
            img = self.transforms(img)

        # Return the processed image & its corresponding target (annotations).
        return img, target

    # The __len__ method returns the total number of samples (images) in the dataset.
    def __len__(self):
        # Return the length of the dataset, i.e., the total number of images.
        return len(self.dataset)

In [ ]:
#@title Transformations & Data Loader

# Define the image transformations (convert images to PyTorch tensors)
# The `transforms.Compose` allows you to chain together multiple transformations.
# Here, we only use `ToTensor`, which converts a PIL Image or a NumPy array (H x W x C)
# in the range [0, 255] to a torch.FloatTensor of shape (C x H x W) in the range [0.0, 1.0].
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert the image to a PyTorch tensor.
])

# Create the training & testing datasets using the custom VOCDataset class.
# - `root` specifies the directory where the Pascal VOC dataset is stored or will be downloaded.
# - `year` specifies the version of the Pascal VOC dataset ('2007' in this case).
# - `image_set` specifies whether to load the 'train' or 'val' split of the dataset.
# - `transforms` specifies the transformations to apply to each image (here, converting to a tensor).
train_dataset = VOCDataset(root='./voc', year='2007', image_set='train', transforms=transform)
test_dataset = VOCDataset(root='./voc', year='2007', image_set='val', transforms=transform)

# Create DataLoaders for the training & testing datasets.
# A DataLoader loads the data in batches & provides various options like shuffling & multi-threaded loading.

# - `train_loader` is for loading the training data:
#   - `train_dataset`: The dataset to load (in this case, the Pascal VOC training set).
#   - `batch_size=4`: Specifies that the data should be loaded in batches of 4 samples.
#   - `shuffle=True`: Randomly shuffle the data at every epoch (helps with better training by preventing the model
#      from learning the order of data).
#   - `collate_fn`: A custom collate function that ensures that the data from each batch is correctly stacked.
#     Here, `tuple(zip(*x))` is used to separate the images & targets (annotations) in the batch.
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))

# - `test_loader` is for loading the testing/validation data:
#   - `test_dataset`: The dataset to load (in this case, the Pascal VOC validation set).
#   - `batch_size=4`: Specifies that the data should be loaded in batches of 4 samples.
#   - `shuffle=False`: Do not shuffle the data; this is typical for validation/testing to ensure consistent evaluation.
#   - `collate_fn`: The same custom collate function is used as in the train_loader.
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))


In [ ]:
#@title Develop the Faster RCNN model

# Load a pre-trained Faster R-CNN model with a ResNet-50 backbone & Feature Pyramid Network (FPN)
# - `fasterrcnn_resnet50_fpn(pretrained=True)` loads a Faster R-CNN model pre-trained on the COCO dataset.
#   - Faster R-CNN is an object detection model that predicts bounding boxes & labels for objects in an image.
#   - ResNet-50 is the backbone used to extract features from the input images.
#   - FPN (Feature Pyramid Network) is used to detect objects at different scales in the image.
# - By using a pre-trained model, we leverage transfer learning, where the model starts with weights learned on a
#   large dataset (COCO), which typically leads to faster convergence & better performance on smaller datasets like Pascal VOC.
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)

# Replace the classifier (box predictor) with a new one that matches the number of classes in the Pascal VOC dataset.
# - The COCO dataset has 80 classes + 1 background class, while Pascal VOC has 20 classes + 1 background class.
# - To adapt the pre-trained Faster R-CNN model to Pascal VOC, we need to replace the final classification layer.

# `num_classes` defines the number of classes that the model should predict.
# - Pascal VOC has 20 object classes, plus 1 background class, so `num_classes` is set to 21.
num_classes = 21  # 20 object classes + 1 background class

# Extract the number of input features to the classifier (box predictor).
# - `in_features` represents the number of input features for the classification layer.
# - We extract this information from the existing `cls_score` layer of the pre-trained model.
in_features = model.roi_heads.box_predictor.cls_score.in_features

# Replace the box predictor (classifier) of the model with a new `FastRCNNPredictor`.
# - `FastRCNNPredictor` is the classification & regression head used in Faster R-CNN.
# - It takes `in_features` as input (the number of features output by the backbone) & predicts `num_classes` classes.
# - By replacing the box predictor, the model can now classify objects based on Pascal VOC's 20 classes.
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)

# Move the model to the appropriate device (GPU if available, otherwise CPU).
# - `model.to(device)` ensures that all model parameters are moved to the same device (GPU/CPU) for efficient computation.
# - This step is crucial before starting the training or inference process.
model.to(device)


In [ ]:
#@title Training Specifications

# Define the optimizer (Stochastic Gradient Descent) & learning rate scheduler

# Get all parameters of the model that require gradients.
# - In PyTorch, only parameters with `requires_grad=True` will have their gradients computed during backpropagation.
# - This list comprehension filters out any parameters that do not require gradients (e.g., frozen layers).
params = [p for p in model.parameters() if p.requires_grad]

# Define the optimizer using Stochastic Gradient Descent (SGD).
# - `optimizer` is responsible for updating the model's parameters based on the gradients computed during backpropagation.
# - `torch.optim.SGD` is a common choice for optimization, particularly for tasks like object detection.
# - `params`: The list of model parameters that will be updated.
# - `lr=0.005`: The learning rate, which controls the step size at each iteration while moving toward a minimum of the
#    loss function.
#   - A smaller learning rate can make training more stable but slower, while a larger learning rate can speed up training
#     but may overshoot minima.
# - `momentum=0.9`: Momentum helps accelerate gradients vectors in the right direction, leading to faster converging.
#   - It’s a technique that helps the optimizer escape local minima & smooth out oscillations.
# - `weight_decay=0.0005`: This adds a regularization term to the loss function (L2 regularization).
#   - It helps prevent the model from overfitting by penalizing large weights.
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# Define the learning rate scheduler.
# - `lr_scheduler` adjusts the learning rate during training based on a predefined schedule.
# - `torch.optim.lr_scheduler.StepLR` reduces the learning rate by a factor of `gamma` every `step_size` epochs.
# - `optimizer`: The optimizer for which to schedule the learning rate.
# - `step_size=3`: The period of learning rate decay, meaning the learning rate will be reduced every 3 epochs.
# - `gamma=0.1`: The multiplicative factor of learning rate decay. After each `step_size`, the learning rate is
#    multiplied by `gamma`.
#   - For example, if the initial learning rate is 0.005, it will become 0.0005 after 3 epochs (0.005 * 0.1).
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)


In [ ]:
#@title Save & Load the Model (if needed)
# Define the file path where you want to save the model
model_save_path = "faster_rcnn_voc.pth"

# Save the model's state dictionary
# - `model.state_dict()` retrieves the model's parameters (weights & biases).
# - `torch.save()` saves these parameters to the specified file.
torch.save(model.state_dict(), model_save_path)

print(f"Model saved to {model_save_path}")


'''
If you need to load the model
'''

# Assume you have the model class defined or imported
# For example, if you are using a Faster R-CNN model:
# model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=False)

# Recreate the model architecture
# model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=False, num_classes=21)  # Adjust `num_classes` as per your model

# Load the saved state dictionary into the model
# model.load_state_dict(torch.load("faster_rcnn_voc.pth"))

# If you are using the model for inference, set it to evaluation mode
# model.eval()

# print("Model loaded successfully!")

In [ ]:
#@title Training the Model

# Set the number of epochs (full passes through the training dataset)
# - `num_epochs` specifies how many times the model will see the entire training dataset.
# - Training for more epochs typically improves model performance but can also increase the risk of overfitting.
num_epochs = 2  # Number of epochs to train

# Start the training loop, which will run for the specified number of epochs.
for epoch in range(num_epochs):

    # Set the model to training mode
    # - `model.train()` tells PyTorch that the model is in training mode.
    # - This is important because certain layers (like dropout & batch normalization)
    #   behave differently during training & evaluation.
    model.train()

    # Initialize a variable to keep track of the running loss
    # - `running_loss` accumulates the loss over each batch within an epoch.
    # - This variable is reset at the start of each epoch.
    running_loss = 0.0

    # Use tqdm to create a progress bar for the training loop
    # - `tqdm(train_loader)` wraps the DataLoader to display a progress bar.
    # - `desc=f"Epoch {epoch + 1}/{num_epochs}"` provides a description that shows
    #    the current epoch number out of the total epochs.
    # - `unit="batch"` specifies that the progress bar units are in batches.
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}", unit="batch")

    # Iterate over the training dataset, loading one batch of data at a time
    # - `enumerate(progress_bar)` allows us to loop over each batch with an index `i`.
    # - `images, targets` represent the data & its corresponding annotations (e.g., bounding boxes & labels).
    for i, (images, targets) in enumerate(progress_bar):

        # Move the images & targets to the specified device (GPU or CPU)
        # - `image.to(device)` moves the image tensor to the device (e.g., GPU) for faster computation.
        # - `targets` are also moved to the device, ensuring that all data is on the same hardware as the model.
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Zero the parameter gradients before the forward pass
        # - `optimizer.zero_grad()` clears the gradients of all optimized parameters.
        # - Gradients are accumulated by default in PyTorch, so they need to be zeroed before each new optimization step.
        optimizer.zero_grad()

        # Perform a forward pass through the model & compute the loss
        # - `model(images, targets)` passes the batch of images & targets through the model.
        # - The model returns a dictionary of losses, one for each component
        #   (e.g., classification loss, bounding box regression loss).
        loss_dict = model(images, targets)

        # Sum up all the individual losses to get the total loss for the batch
        # - `losses` is the total loss that will be used to update the model's parameters.
        losses = sum(loss for loss in loss_dict.values())

        # Perform the backward pass & optimize the model parameters
        # - `losses.backward()` computes the gradient of the loss with respect to model parameters.
        # - `optimizer.step()` updates the model parameters using the computed gradients.
        losses.backward()
        optimizer.step()

        # Update running loss
        # - `running_loss` accumulates the total loss for the current epoch.
        running_loss += losses.item()

        # Calculate & display the average loss so far
        # - `avg_loss` is the average loss for the current epoch up to the current batch.
        # - `progress_bar.set_postfix(loss=avg_loss)` updates the tqdm progress bar with the current average loss.
        avg_loss = running_loss / (i + 1)
        progress_bar.set_postfix(loss=f"{avg_loss:.5f}")

    # Update the learning rate using the scheduler
    # - `lr_scheduler.step()` adjusts the learning rate based on the learning rate schedule defined earlier.
    # - This typically reduces the learning rate after a set number of epochs to help the model converge more smoothly.
    lr_scheduler.step()



In [ ]:
#@title Evaluate the Model

# Import necessary libraries
from torchvision.ops import nms  # Import Non-Maximum Suppression (NMS) from torchvision

# Set the model to evaluation mode
# - `model.eval()` tells PyTorch that the model is in evaluation (inference) mode.
# - This is important because certain layers (like dropout & batch normalization) behave differently during training & evaluation.
# - For example, dropout is disabled during evaluation, & batch normalization uses running statistics rather than batch statistics.
model.eval()

# Get a dictionary that converts class numbers to actual item names.
# - `VOC_CLASSES` is assumed to be a dictionary that maps item names to class numbers.
# - `class_to_item` reverses this mapping, making it easier to label the predictions with item names.
class_to_item = {value: key for key, value in VOC_CLASSES.items()}

# Define a function to visualize predictions made by the model
# - This function takes an image, the predicted bounding boxes, & the corresponding labels, & then displays the image with the predictions.
def visualize_predictions(image, boxes, labels):
    # Rearrange the dimensions of the image tensor from (C, H, W) to (H, W, C)
    # - `permute(1, 2, 0)` changes the order of dimensions so that the image can be converted to a format suitable for visualization.
    img = image.permute(1, 2, 0).cpu().numpy()

    # Rescale the pixel values from [0, 1] to [0, 255] & convert to unsigned 8-bit integers
    # - This is necessary because images are often normalized in preprocessing, & we need to convert them back for visualization.
    img = (img * 255).astype(np.uint8)

    # Convert the NumPy array to a PIL Image
    # - `Image.fromarray(img)` creates a PIL image from the NumPy array, which allows us to draw on it.
    img = Image.fromarray(img)

    # Create a drawing context to draw on the image
    # - `ImageDraw.Draw(img)` allows us to draw shapes & text on the image.
    draw = ImageDraw.Draw(img)

    # Iterate over each bounding box & label pair
    for box, label in zip(boxes, labels):
        # Draw a rectangle (bounding box) on the image
        # - `box.tolist()` converts the bounding box tensor to a list format.
        # - `outline="red"` specifies the color of the bounding box.
        # - `width=3` sets the thickness of the bounding box.
        draw.rectangle(box.tolist(), outline="red", width=3)

        # Define the text to be displayed
        text = class_to_item[label.item()]

        # Calculate the size of the text using `textbbox`
        # - `textbbox` returns the bounding box of the text in the format (left, top, right, bottom).
        text_bbox = draw.textbbox((box[0], box[1]), text)

        # Draw a rectangle behind the text to serve as a background
        # - The rectangle's size is based on the text's bounding box, with a small padding added.
        draw.rectangle(
            [text_bbox[0], text_bbox[1], text_bbox[2] + 4, text_bbox[3] + 4],
            fill="gray"  # Gray background for the text
        )

        # Draw the label text on top of the gray rectangle
        # - `box[0] + 2` & `box[1] + 2` are offsets to place the text within the gray rectangle.
        draw.text((box[0] + 2, box[1] + 2), text, fill="white")

    # Display the image with bounding boxes & labels using matplotlib
    # - `plt.imshow(img)` shows the image with the drawn bounding boxes & labels.
    #plt.figure(figsize = [5,5])
    plt.imshow(img)

    # Turn off the axis labels for a cleaner display
    plt.axis('off')

    # Display the image in a window
    plt.show()

# Run the model on the test dataset
# - The `torch.no_grad()` context manager is used to disable gradient computation.
# - This is essential during evaluation because it reduces memory consumption & speeds up the computation, as gradients are not needed.
with torch.no_grad():
    # Iterate over the test dataset in batches
    # - `test_loader` provides batches of images & their corresponding ground truth annotations.
    for images, targets in test_loader:

        # Move the images to the appropriate device (GPU or CPU)
        # - This ensures that the data is on the same device as the model for computation.
        images = list(image.to(device) for image in images)

        # Perform a forward pass through the model to get predictions
        # - `model(images)` returns the predicted bounding boxes, labels, & confidence scores for each image in the batch.
        outputs = model(images)

        # Iterate over each image in the batch to visualize the predictions
        for i, image in enumerate(images):
            # Extract the predicted bounding boxes & labels for the current image
            boxes = outputs[i]['boxes']
            labels = outputs[i]['labels']
            scores = outputs[i]['scores']  # Get confidence scores for each box

            # Apply Non-Maximum Suppression (NMS)
            # - `nms(boxes, scores, iou_threshold=0.5)` filters out overlapping boxes with lower confidence scores.
            # - `iou_threshold=0.5` means that boxes with an IoU (Intersection over Union) greater than 0.5 will be considered overlapping.
            keep = nms(boxes, scores, iou_threshold=0.5)

            # Keep only the boxes & labels that survived NMS
            boxes = boxes[keep]
            labels = labels[keep]

            # Call the `visualize_predictions` function to draw the bounding boxes & labels on the image
            visualize_predictions(image, boxes, labels)

        # Break the loop after the first batch to visualize only the first set of predictions
        break  # Visualize the first batch only

# Auto runtime disconnection to save CPU/GPU/TPU allocations
from google.colab import runtime
import time
time.sleep(5)
runtime.unassign()


## **2. Experiment with Traffic Sign Dataset & YOLO**

### **2.1. Traffic Sign Dataset**

> [The Traffic Signs Dataset in YOLO format](https://www.kaggle.com/datasets/valentynsichkar/traffic-signs-dataset-in-yolo-format) is a relatively small dataset designed for training & testing object detection models, particularly in recognizing & localizing limited traffic signs categories in images. This dataset is a small version of a more comprehensive traffic sign dataset, which is crucial for developing computer vision systems used in autonomous vehicles, traffic monitoring, & advanced driver assistance systems (ADAS).

> The dataset includes four traffic sign categories (prohibitory, dangerous, mandatory, & other) & limited images captured in diverse conditions. A comprehensive dataset would ensure the inclusion of a variety of images with different lighting, weather, & traffic environments, ensuring that models trained on this data can generalize well to real-world scenarios. Each image is accompanied by a corresponding label file in YOLO format, where the annotations include the class of the traffic sign & the normalized coordinates of the bounding box that encloses the sign. A comprehensive dataset may encompass a wide range of sign types, such as stop signs, speed limits, & warning signs, providing a comprehensive training ground for object detection models.

> In the YOLO format, each annotation file for an image contains one or more lines, with each line representing a single object. The format of each line is as follows: `<class_id> <x_center> <y_center> <width> <height>`, where all coordinates are normalized to the width & height of the image, & the class ID corresponds to a specific traffic sign class. This format is particularly efficient for real-time object detection tasks, as it allows the YOLO model to quickly predict & localize multiple objects within an image with a single forward pass. Using a Traffic Signs Dataset in YOLO format enables the development of robust, fast, & accurate models that are crucial for applications requiring real-time detection & response, such as in autonomous driving systems where the timely recognition of traffic signs can be critical to safety & navigation.

> The [Ultralytics](https://www.ultralytics.com/) library is a powerful, user-friendly Python package designed for developing & deploying cutting-edge deep learning models, particularly for object detection tasks using the YOLO (You Only Look Once) framework. It offers an easy-to-use interface for training, validating, & deploying YOLO models, including the latest YOLOv5 & YOLOv8 versions, which are known for their accuracy & speed in detecting objects in images & videos. The library is built with simplicity & efficiency in mind, providing extensive features such as automated hyperparameter tuning, model export to various formats (e.g., ONNX, TensorRT), & integration with popular deep learning frameworks like PyTorch. Ultralytics is widely used in both academic research & industry applications, enabling developers to build & deploy robust object detection systems with minimal effort.

> To employ the YOLO models of Ultralytics library, Your dataset should be in the "YOLO format," which includes:
- Images: Organized in a directory.
- Labels: Corresponding text files containing the bounding box coordinates & class labels.

> Ensure your dataset structure looks like this:
```
./dataset/your_data_name/
    images/
        image1.jpg
        image2.jpg
        ...
    labels/
        image1.txt
        image2.txt
        ...
    train.txt (list of paths to training images)
    valid.txt (list of paths to validation images)
```

> If your label data is not in YOLO format, you might need to convert it. Each .txt **label** file should have the format:
```
<class_id> <x_center> <y_center> <width> <height>
```

> Furthermore, you need to define the dataset configuration in a YAML file at ```./dataset/your_data_name/your_data_name.yaml```,  specifying paths to your images & labels. Here is an example:
```
train: ./dataset/your_data_name/train.txt
val: ./dataset/your_data_name/val.txt
nc: 7  # number of classes
names: ['class1', 'class2', ...]  # class names
```


In [ ]:
#@title Install Ultralytics & Download the Data
'''
Runtime: GPU $$$
'''

# Install the Ultralytics library, which includes the YOLO (You Only Look Once) framework
# & tools for developing & deploying object detection models.
!pip install ultralytics

# Download the traffic dataset from a provided URL. The dataset is compressed into a .zip file.
# The dataset contains images & annotations for training or testing object detection models.
!wget https://storage.googleapis.com/535743/module09/traffic.zip

# Unzip (extract) the contents of the downloaded traffic.zip file into a directory named 'dataset'.
# The '-q' flag is used to suppress the output of the unzip command to keep the output cleaner.
!unzip -q ./traffic.zip -d ./dataset


In [ ]:
#@title Create the YAML File

# The `%%writefile` magic command writes the following content into a file named `traffic.yaml`
# located in the `./dataset/traffic/` directory. If the directory or file does not exist,
# it will be created.

%%writefile ./dataset/traffic/traffic.yaml

# Define the paths to the dataset for training & validation.
# 'train' specifies the path to the file listing all training images.
train: /content/dataset/traffic/train.txt  # Path to the images used for training

# 'val' specifies the path to the file listing all validation images.
val: /content/dataset/traffic/test.txt   # Path to the images used for validation

# 'nc' defines the number of classes that the model needs to detect.
nc: 4  # Number of classes

# 'names' lists the names of each class, corresponding to the number of classes (nc).
# These are the labels that the model will learn to recognize.
names: ['prohibitory', 'danger', 'mandatory', 'other']

### **2.2. Training Using YOLO V5s**

In [ ]:
#@title Load the Model

# Import the YOLO class from the Ultralytics library, which contains implementations
# for various YOLO models (including YOLOv5 & YOLOv8).
from ultralytics import YOLO

# Load a pre-trained YOLOv5 model.
# The "yolov5s.pt" file is a lightweight version of the YOLOv5 model, known as YOLOv5s ("s" for small).
# This model is loaded with pre-trained weights, making it ready for inference or fine-tuning.
model = YOLO("yolov5s.pt")

# Print the architecture of the model to verify that it has been loaded correctly.
# This can include details like the layers, parameters, & output shapes.
# print(model)


In [ ]:
#@title Train the Model
# Train the YOLO model on the custom dataset.
# The `train` method initiates the training process for the model using the specified parameters.

model.train(
    data="./dataset/traffic/traffic.yaml",  # Path to the YAML file containing dataset information.
    epochs=5,                               # Number of training epochs, or how many times the model will see the entire dataset.
    batch=16,                               # Batch size, or the number of images the model processes before updating its weights.
    imgsz=640                               # Image size (resolution) that the model will resize input images to during training.
)


In [ ]:
#@title Evaluate the Model

# The `val` method is used to evaluate the performance of the YOLO model on the validation dataset.

# When you run `model.val()` in the Ultralytics YOLO framework, it initiates the
# evaluation process on the validation dataset. During this process, the model
# generates a set of results that are automatically saved to a newly created folder,
# typically named `runs/val/exp` (or `expn` if there are multiple evaluations).
# This folder contains various files & subdirectories that provide detailed
# insights into the model's performance. Among the key contents are images that
# have been annotated with the predicted bounding boxes, class labels, & confidence
# scores, which allow you to visually inspect how well the model is detecting and
# classifying objects. The folder also contains text files or logs that include
# performance metrics such as precision, recall, & mAP (mean Average Precision)
# scores for each class, which are crucial for quantitatively assessing the model's
# accuracy & effectiveness.

# Additionally, the folder may include confusion matrices & other summary statistics
# that offer deeper insights into the model's performance, such as how often the model
# confuses one class with another. These results help in diagnosing potential issues,
# such as specific classes that are underperforming or images where the model consistently
# fails to detect objects. By analyzing the content of this folder, you can make informed
# decisions about whether further training, hyperparameter tuning, or dataset adjustments
# are necessary to improve the model's performance before deploying it in a real-world
# scenario. This detailed feedback loop is essential for refining object detection models
# to achieve optimal accuracy & reliability.

model.val()  # This will automatically use the validation dataset specified during training.



In [ ]:
#@title Evaluate A Custom Image

# Import necessary libraries for plotting & image handling
import matplotlib.pyplot as plt
import matplotlib.patches as patches  # Used to draw bounding boxes on the image
from PIL import Image  # To handle image opening & manipulation

# The locale library in Python is a module that provides a way to deal with
# locale-specific data & operations. A locale is a set of parameters that defines
# the user’s language, region, & any special variant preferences, such as date
# formats, number formats, or currency symbols. The locale module allows Python
# programs to adapt to different cultural norms & conventions when processing
# data that may vary by region or language.
import locale

# As of Aug 2024, after using YOLO from ultralytics libraries in Google Colab, we
# might encounter some kind of command line error that here we can fix it:

# Force Python to use UTF-8 encoding for all text operations.
locale.getpreferredencoding = lambda: "UTF-8"

# Adjust the locale settings to ensure proper encoding handling
# This ensures that in Colab, the ! mark works for command line scripts
# The error `NotImplementedError: A UTF-8 locale is required. Got ANSI_X3.4-1968`
# occurs when the system's default text encoding is set to an outdated or incompatible
# format like `ANSI_X3.4-1968`, which is essentially ASCII & lacks support for modern
# characters. This can cause issues when Python expects UTF-8 encoding, which is now
# the standard for handling diverse character sets. To resolve this, you can override
# Python's preferred encoding by using `locale.getpreferredencoding = lambda: "UTF-8"`,
# which forces Python to use UTF-8 for all text operations within that session, ensuring
# compatibility with modern applications & installations. This is a useful workaround
# in environments where you cannot easily change the system's locale settings.


# Download the test image from a specified URL
!wget https://storage.googleapis.com/535743/module09/test.png

# Specify the path to the downloaded image
image_path = '/content/test.png'

# Use the YOLO model to perform inference on the specified image
# This will generate results including detected bounding boxes, classes, & confidence scores
results = model(image_path)

# Create a figure & axis for plotting without any borders
fig, ax = plt.subplots(1)
image = Image.open(image_path)  # Open the image using PIL

# Display the image on the plot, setting it as the background
ax.imshow(image)

# Extracting bounding boxes, class labels, & confidence scores from the results
for result in results:
    boxes = result.boxes.xyxy  # Bounding box coordinates (x1, y1, x2, y2)
    confidences = result.boxes.conf  # Confidence scores for each detected object
    class_ids = result.boxes.cls  # Class IDs for each detected object

    # Define the class names corresponding to the class IDs
    # This list should match the classes the model was trained on
    class_names = ['class1', 'class2', 'class3', 'class4']  # Replace with actual class names

    # Plot each bounding box on the image
    for i in range(len(boxes)):
        box = boxes[i].cpu().numpy()  # Convert the bounding box coordinates from tensor to numpy array
        confidence = confidences[i]  # Get the confidence score
        class_id = int(class_ids[i])  # Get the class ID & convert it to an integer

        # Create a Rectangle patch to represent the bounding box
        rect = patches.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                                 linewidth=2, edgecolor='red', facecolor='none')
        # Add the bounding box to the plot
        ax.add_patch(rect)

        # Create a label with the class name & confidence score
        label = f"{class_names[class_id]}: {confidence:.2f}"
        # Place the label above the bounding box on the plot
        plt.text(box[0], box[1] - 10, label, color='red', fontsize=12, weight='bold')

# Turn off axis numbers & ticks for a cleaner look
ax.axis('off')

# Save the final image with the bounding boxes & labels to a file
fig.savefig('test_output.png', bbox_inches='tight', pad_inches=0)

# Close the figure to free up memory & avoid potential memory leaks
plt.close(fig)

# <font color="#418FDE" size="6.5" uppercase>**B: Object Detection Experiments**</font>
----

In this lecture, you learned to:

* Develop fine-tuned object detection models such as Faster RCNN & YOLO.

In the next Module (Module 10), we go over "Object Detection, Part 2".